# 14 MRI文件接收、技术质控与ROI边界

本步获准继续。固定规则见`reports/ucsd_mri_intake_protocol.md`。两例仅为技术样本，不估计性能。缺少文件时实际执行程序并输出blocked，不伪造影像、ROI或AUC。下列代码不读取临床标签，不生成模拟影像。

In [1]:
from pathlib import Path
import sys,json,hashlib
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from rn_public.ucsd_mri_intake import run,CASES,SEQUENCES
result=run(ROOT)
s=result['summary']
print(json.dumps(s,ensure_ascii=False,indent=2))

{
  "checked_at_utc": "2026-10-29T04:37:15.554208+00:00",
  "input_directory_exists": false,
  "expected_files": 12,
  "missing_expected_files": 12,
  "uniquely_located_expected_files": 0,
  "decoded_expected_files": 0,
  "technical_pass_files": 0,
  "minimal_FLAIR_batch_technical_pass": false,
  "expanded_batch_grid_screen_pass": false,
  "source_authenticity_verified": false,
  "anatomical_alignment_verified": false,
  "ROI_for_prediction_approved": false,
  "MRI_features_extracted": false,
  "MRI_model_fitted": false,
  "clinical_models_refitted": false,
  "nibabel_version": "5.4.2",
  "numpy_version": "2.3.5",
  "thresholds": {
    "max_bytes": 536870912,
    "max_voxels": 40000000,
    "corner_delta_mm": 0.1
  },
  "scope": "Two selected technical cases only; no labels read by QC; no clinical or ROI acceptance",
  "state": "blocked_missing_or_failed_sample_files",
  "protocol_sha256": "177d8e86eaef28815e0309e325a5b55e89a6a56d2427430c17fc87a77f75e632",
  "module_sha256": "b9da93e85

## 逐文件状态

12行是两例扩展清单，不代表12份影像已到位；其中两份FLAIR是最小批次。重复文件、病例目录冲突及链接均拦截。

In [2]:
checklist=pd.read_csv(ROOT/'reports/ucsd_mri_intake/sample_file_checklist.csv')
assert len(checklist)==len(CASES)*len(SEQUENCES)==12
assert checklist.expected_filename.is_unique
assert checklist.minimal_batch.sum()==2
display(checklist[['ID','sequence','minimal_batch','status','decoded','technical_pass']])

,ID,sequence,minimal_batch,status,decoded,technical_pass
0,UCSD-PTGBM-0001_01,FLAIR,True,missing,False,False
1,UCSD-PTGBM-0001_01,T1pre,False,missing,False,False
2,UCSD-PTGBM-0001_01,T1post,False,missing,False,False
3,UCSD-PTGBM-0001_01,T2,False,missing,False,False
4,UCSD-PTGBM-0001_01,ADC_vendor,False,missing,False,False
5,UCSD-PTGBM-0001_01,BraTS_tumor_seg,False,missing,False,False
6,UCSD-PTGBM-0003_01,FLAIR,True,missing,False,False
7,UCSD-PTGBM-0003_01,T1pre,False,missing,False,False
8,UCSD-PTGBM-0003_01,T1post,False,missing,False,False
9,UCSD-PTGBM-0003_01,T2,False,missing,False,False


## 完整体素及几何读取的结果

NiBabel头部载入是延迟读取；实际质控需gzip全流CRC及完整缩放体素读取。技术通过不等于解剖内容配准通过。下面只显示真实接收文件结果；没有文件时表格为空。

In [3]:
located=[r for r in result['files'] if r['status']!='missing']
display(pd.DataFrame(located))
print('Actual expected files decoded:',s['decoded_expected_files'])
print('Minimal FLAIR batch technical pass:',s['minimal_FLAIR_batch_technical_pass'])
assert s['decoded_expected_files']==sum(r['decoded'] for r in result['files'])

Actual expected files decoded: 0
Minimal FLAIR batch technical pass: False


""


## ROI与来源证据门槛

TCT/ECT/NECT参考分割禁止作为预测输入；普通BraTS也尚未确认盲法及标签含义，不自动授权预测ROI。SHA256只是本地内容指纹，没有官方逐文件哈希，不能证明来源。实际图像及后续审阅未完成时不建立MRI模型。

In [4]:
print('Excluded inputs:',json.dumps(result['excluded_files'],ensure_ascii=False,indent=2))
assert s['ROI_for_prediction_approved'] is False
assert s['MRI_features_extracted'] is False and s['MRI_model_fitted'] is False
assert s['clinical_models_refitted'] is False
protocol=ROOT/'reports/ucsd_mri_intake_protocol.md'
assert s['protocol_sha256']==hashlib.sha256(protocol.read_bytes()).hexdigest()
print('Current state:',s['state'])

Excluded inputs: []
Current state: blocked_missing_or_failed_sample_files


## 软件验证与真实研究数据分开

自动测试使用临时小数组注入截断文件、非有限值、坐标冲突、空掩膜等错误；临时测试文件不计为实际UCSD影像。软件测试回执可复核。当前仍需真实文件，下一研究阶段等待用户批准。

In [5]:
receipt=ROOT/'reports/ucsd_mri_intake/software_test_receipt.json'
if receipt.exists():
    print(receipt.read_text())
else:
    print('Software tests must be run separately; no test success claimed by this notebook.')
print('No clinical model or MRI diagnostic performance estimated in this step.')

{
  "checked_at_utc": "2026-10-29T04:37:14.040888+00:00",
  "command": "python -m unittest discover -s tests -v",
  "tests_run": 57,
  "failures": 0,
  "errors": 0,
  "skipped": 0,
  "success": true,
  "elapsed_seconds": 4.154,
  "new_MRI_intake_test_cases": 10,
  "synthetic_arrays_are_software_fixtures_only": true,
  "real_UCSD_images_validated_by_tests": 0
}

No clinical model or MRI diagnostic performance estimated in this step.
